# Using the Allmaps API with the Digital Commonwealth JSON API

This notebook shows how to retrieve object metadata for a map in Digital Commonwealth / LMEC by climbing up from the Allmaps API to the Digital Commonwealth JSON API.

Let's say we have this set of Allmaps XYZ tiles in a web map:
    
    https://allmaps.xyz/maps/10c1664f2c9d2cff/{z}/{x}/{y}.png

To determine where these tiles came from, we can use the Allmaps API in combination with the Digital Commonwealth JSON API to programmatically climb from the map ID `10c1664f2c9d2cff` all the way up to the Digital Commonwealth manifest for the parent object. Metadata for these tiles, including author, title, date, and more, are all available with the parent object.

This will happen in x steps:

1. get the Allmaps ID from the XYZ tiles
2. pass the Allmaps ID to the Allmaps API's base URL to return the DC image ID
3. pass the DC image ID to the DC JSON API's base URL to return its parent object's manifest ID
4. pass manifest ID to LMEC collections base URL to return object metadata

# 1. get the Allmaps ID from the XYZ tiles

In [91]:
import requests
import json

allmapsId = requests.get("https://api.allmaps.org/maps/10c1664f2c9d2cff/manifests").json()[0]['imageId']

print(allmapsId)

c6ecdd86e3334373


# 2. pass the Allmaps ID to the Allmaps API's base URL to return the DC image ID

In [92]:
baseAnnotationURL = "https://annotations.allmaps.org/images/"
fullAnnotationURL = baseAnnotationURL+allmapsId
annotationRequest = requests.get(fullAnnotationURL)
imageDC = annotationRequest.json()['items'][0]['target']['service'][0]['@id']
imageIdDC = (imageDC[-9:])

print(imageIdDC)

4742cs815


# 3. pass the DC image ID to the DC JSON API's base URL to return its parent object's manifest ID

In [93]:
imageParent = requests.get("https://www.digitalcommonwealth.org/search/commonwealth:" + imageIdDC + ".json")
manifestId = imageParent.json()['data']['attributes']['is_file_set_of_ssim'][0] + "/manifest.json"

print(manifestId)

commonwealth:6t055z60n/manifest.json


# 4. pass manifest ID to LMEC collections base URL to return object metadata

In [94]:
collectionsRecordManifest = "https://collections.leventhalmap.org/search/" + manifestId
metadataRequest = requests.get(collectionsRecordManifest)
metadata = json.dumps(metadataRequest.json(), indent=2)

print(metadata)

{
  "@context": "http://iiif.io/api/presentation/2/context.json",
  "@id": "https://ark.digitalcommonwealth.org/ark:/50959/6t055z60n/manifest",
  "@type": "sc:Manifest",
  "label": "South End renewal area",
  "thumbnail": {
    "@id": "https://ark.digitalcommonwealth.org/ark:/50959/6t055z60n/thumbnail",
    "service": {
      "@context": "http://iiif.io/api/image/2/context.json",
      "@id": "https://iiif.digitalcommonwealth.org/iiif/2/commonwealth:4742cs815",
      "profile": "http://iiif.io/api/image/2/level2.json"
    }
  },
  "viewingHint": "individuals",
  "metadata": [
    {
      "label": "Title",
      "value": "South End renewal area : existing conditions & proposed treatment areas"
    },
    {
      "label": "Date",
      "value": "October 1960"
    },
    {
      "label": "Creator",
      "value": "Boston Redevelopment Authority"
    },
    {
      "label": "Publisher",
      "value": "Boston : Boston Redevelopment Authority"
    },
    {
      "label": "Type of Resource",